# 1. SAP AI Core ontology generation with Semantica

Design-time workflow: SAP AI Core is called through LiteLLM using `sap/<model>`. Semantica's `LLMOntologyGenerator` and `OWLGenerator` handle ontology generation and Turtle export. The result is a proposal; it does not replace the curated runtime ontology or update the mapper.

## Configure credentials outside the notebook

Set `AICORE_SERVICE_KEY` and `AICORE_RESOURCE_GROUP` in your shell or secret manager. Do not paste credentials into notebook cells.

In [ ]:
import os
assert os.getenv("AICORE_SERVICE_KEY"), "Set AICORE_SERVICE_KEY first"
assert os.getenv("AICORE_RESOURCE_GROUP"), "Set AICORE_RESOURCE_GROUP first"
model = "sap/gpt-4o"

## Inspect the source structure

In [ ]:
from northstar_authz.service import load_data
from northstar_authz.ontology_generation import build_generation_prompt

data = load_data()
print(build_generation_prompt(data))

## Generate a proposed ontology

The SAP adapter bridges the model call through Semantica's LiteLLM wrapper. This POC does not implement OAuth or HTTP transport.

In [ ]:
from northstar_authz.ontology_generation import generate_ontology

proposal = generate_ontology(
    source_path="data/northstar.json",
    output_path="generated/northstar.proposed.ttl",
    model=model,
)
print("Proposal written to generated/northstar.proposed.ttl")

## Review ontology changes before adoption

In [ ]:
from northstar_authz.ontology_diff import diff_ontologies

diff = diff_ontologies("ontology/northstar.ttl", "generated/northstar.proposed.ttl")
diff

**Takeaway:** Semantica proposes ontology structure. It does not generate executable source-to-KG mappings. The curated ontology remains authoritative until a reviewed migration updates the mapper and validation contract.